# 07 — R1 DL data preparation
Only MH_01_easy and MH_02_easy. This notebook loads prepared data; run `python scripts/prepare_r1_data.py` first. No models are trained.

In [1]:
from pathlib import Path
import sys, json
ROOT = Path.cwd()
if ROOT.name == "notebooks": ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "scripts"))
from demo_dl_data import demonstrate
from vio_drift.data.dl_dataset import DriftDataset, make_dataloader
from vio_drift.data.augment import IMUAugmentation


## Splits and tensor contract
Final ceil(20%) validation; last training interval ends at least four seconds before validation starts. Scalers and physics calibration use inner training only. Physics retains the Phase 1 ground-truth initial-state assumption.

In [2]:
rows = demonstrate()
assert len(rows) == 6


{'fold': 'leave_MH_01_easy', 'split': 'train', 'windows': 450, 'shapes': {'imu': [32, 6, 400], 'feats': [32, 104], 'phys': [32, 4], 'y_mag': [32], 'y_vec': [32, 3]}}
{'fold': 'leave_MH_01_easy', 'split': 'validation', 'windows': 119, 'shapes': {'imu': [32, 6, 400], 'feats': [32, 104], 'phys': [32, 4], 'y_mag': [32], 'y_vec': [32, 3]}}
{'fold': 'leave_MH_01_easy', 'split': 'test', 'windows': 720, 'shapes': {'imu': [32, 6, 400], 'feats': [32, 104], 'phys': [32, 4], 'y_mag': [32], 'y_vec': [32, 3]}}
{'fold': 'leave_MH_02_easy', 'split': 'train', 'windows': 553, 'shapes': {'imu': [32, 6, 400], 'feats': [32, 104], 'phys': [32, 4], 'y_mag': [32], 'y_vec': [32, 3]}}


{'fold': 'leave_MH_02_easy', 'split': 'validation', 'windows': 144, 'shapes': {'imu': [32, 6, 400], 'feats': [32, 104], 'phys': [32, 4], 'y_mag': [32], 'y_vec': [32, 3]}}
{'fold': 'leave_MH_02_easy', 'split': 'test', 'windows': 592, 'shapes': {'imu': [32, 6, 400], 'feats': [32, 104], 'phys': [32, 4], 'y_mag': [32], 'y_vec': [32, 3]}}


## Deterministic training observations
Call `set_epoch(epoch)` before each epoch. Crop means edge observation masking on the unchanged full timeline, not a shorter target interval. Features, physics and labels remain original reference modalities.

In [3]:
ds = DriftDataset(ROOT/"data/processed", "leave_MH_01_easy", "train", augmentation=IMUAugmentation(gaussian_noise=True, random_crop=True))
a = next(iter(make_dataloader(ds, seed=42)))
b = next(iter(make_dataloader(ds, seed=42)))
import torch
assert torch.equal(a["imu"], b["imu"])
print("Same-seed DataLoader replay passed", tuple(a["imu"].shape))


Same-seed DataLoader replay passed (32, 6, 400)


## Camera and experiments
Camera-off omits `cam`. Camera-on requires R3 uint8 arrays and a key sidecar in windows CSV order. Synthetic camera behavior is exercised by tests. A1/A2 model results await R2/R5; manifests below contain preparation only. A2 fractions refer to eligible inner training, with fixed validation/test.

In [4]:
manifest = json.loads((ROOT/"results/ablations/r1_data_manifest.json").read_text())
print(manifest["status"], len(manifest["jobs"]), "jobs")
for row in manifest["counts"]: print(row)


data_ready_models_not_run 84 jobs
{'fraction': 0.5, 'fold_id': 'leave_MH_01_easy', 'train': 225, 'validation': 119, 'test': 720}
{'fraction': 0.5, 'fold_id': 'leave_MH_02_easy', 'train': 277, 'validation': 144, 'test': 592}
{'fraction': 0.75, 'fold_id': 'leave_MH_01_easy', 'train': 338, 'validation': 119, 'test': 720}
{'fraction': 0.75, 'fold_id': 'leave_MH_02_easy', 'train': 415, 'validation': 144, 'test': 592}
{'fraction': 1.0, 'fold_id': 'leave_MH_01_easy', 'train': 450, 'validation': 119, 'test': 720}
{'fraction': 1.0, 'fold_id': 'leave_MH_02_easy', 'train': 553, 'validation': 144, 'test': 592}
